# UNet 训练（Kaggle 快速版 · 60 epoch · ~1.5 h）

**前置条件**：
- Settings: GPU **T4 x2** + Internet **On** + Persistence **Files only**
- Inputs 必须挂载：`rectal-cancer-data` + `ctai-code-and-splits`（最新版）

**优化点**（相比原 200 epoch 默认配置）：
- epochs 200→60，repeats 50→20，eval_interval 5→10，full_eval_interval 50→999
- batch_size 4→8，num_workers 0→4（DataLoader 单线程是之前的主要瓶颈）
- 预计单 epoch ~1.5 min，总训练 ~1.5 h，安全在 12h Kaggle session 上限内

**Run All 之前**：再次确认 SMOKE_TEST = False（Cell 3）。

In [ ]:
# === Cell 1: 环境 + 依赖检查 ===
import os, sys, glob, subprocess, shutil
print('python  :', sys.version.split()[0])
print('cwd     :', os.getcwd())

# Kaggle 已预装所需依赖（SimpleITK 2.5.3 / albumentations 2.0.8 / pydicom 3.0.1
# / torch 2.10+cu128 / cv2 4.13），跳过 pip install 避免 DNS 失败。
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
#                        'SimpleITK', 'albumentations==1.4.0', 'pydicom'])
print('[INFO] Skipped pip install — using Kaggle pre-installed packages')

import torch
print('torch   :', torch.__version__,
      '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), '[FATAL] CUDA 不可用，请在 Settings 里选 GPU T4 x2 后重启 session'

In [ ]:
# === Cell 2: 路径解析 ===
# Kaggle 把这个账号的 dataset 挂在异常前缀 /kaggle/input/datasets/ramyaramyarao/
# 下，所以候选列表把它放最前面，常规挂载点作后备。
import os, sys, json, hashlib

KAGGLE_NS = '/kaggle/input/datasets/ramyaramyarao'

DATA_CANDIDATES = [
    f'{KAGGLE_NS}/rectal-cancer-data/直肠癌数据',
    f'{KAGGLE_NS}/rectal-cancer-data',
    '/kaggle/input/rectal-cancer-data/直肠癌数据',
    '/kaggle/input/rectal-cancer-data',
]

CODE_CANDIDATES = [
    f'{KAGGLE_NS}/ctai-code-and-splits/ctai_model_code',
    '/kaggle/input/ctai-code-and-splits/ctai_model_code',
]

def _pick(cands, kind):
    for c in cands:
        if os.path.isdir(c):
            return c
    raise SystemExit(f'[FATAL] 找不到 {kind} 目录。请检查 Input 面板是否挂载了正确的 dataset。\n'
                     f'尝试过的候选路径：\n  ' + '\n  '.join(cands))

DATA_DIR = _pick(DATA_CANDIDATES, 'rectal-cancer-data')
SRC_CODE = _pick(CODE_CANDIDATES, 'ctai-code-and-splits')

# 统计患者数（粗略校验数据完整性）
patient_dirs = [d for d in os.listdir(DATA_DIR)
                if os.path.isdir(os.path.join(DATA_DIR, d)) and d.isdigit()]
print(f'DATA_DIR  = {DATA_DIR}  (患者数 {len(patient_dirs)})')
print(f'SRC_CODE  = {SRC_CODE}')

# 校验 splits.json fingerprint
SPLITS_SRC = os.path.join(SRC_CODE, 'splits.json')
with open(SPLITS_SRC, 'r', encoding='utf-8') as f:
    sp = json.load(f)
EXPECTED_FP = '84706e8b75c8f403'
fp = sp.get('fingerprint')
if fp != EXPECTED_FP:
    raise SystemExit(f'[FATAL] splits fingerprint 不匹配：got {fp!r} expected {EXPECTED_FP!r}')
tr, va, te = (len(sp['splits'][k]) for k in ('train', 'val', 'test'))
print(f'splits OK : fp={fp}, train={tr} val={va} test={te}')

In [ ]:
# === Cell 3: 训练开关 ===
# False = 正式训练（60 epoch，~1.5 h，T4 x2）
# True  = smoke test（5 epoch，~10 min，仅验证 pipeline）
SMOKE_TEST = False
print('SMOKE_TEST =', SMOKE_TEST)

In [ ]:
# === Cell 3.5: 拷贝代码到可写目录 + 改快配置 ===
# Kaggle dataset 是只读 fuse 挂载，无法直接改 config.py 里的默认值。
# train_unet.py 只透传 epochs/batch_size/num_workers 三个 CLI 参数，
# repeats / eval_interval / full_eval_interval / save_interval 必须改源码默认值。
# 拷贝整个 ctai_model_code 到 /kaggle/working/ 再改。
import shutil, os, re, subprocess

DST_CODE = '/kaggle/working/ctai_model_code'
if os.path.exists(DST_CODE):
    shutil.rmtree(DST_CODE)
shutil.copytree(SRC_CODE, DST_CODE)
print(f'[copy] {SRC_CODE} -> {DST_CODE}')

cfg_path = os.path.join(DST_CODE, 'CTAI_model', 'config.py')
with open(cfg_path, 'r', encoding='utf-8') as f:
    code = f.read()

# 关键：把会拖慢训练的几个 dataclass 默认值改小
patches = [
    (r'repeats:\s*int\s*=\s*\d+',            'repeats: int = 20'),
    (r'eval_interval:\s*int\s*=\s*\d+',      'eval_interval: int = 10'),
    (r'full_eval_interval:\s*int\s*=\s*\d+', 'full_eval_interval: int = 999'),
    (r'save_interval:\s*int\s*=\s*\d+',      'save_interval: int = 10'),
]
for pat, rep in patches:
    new_code, n = re.subn(pat, rep, code)
    assert n == 1, f'patch failed: {pat} (matched {n} times)'
    code = new_code

with open(cfg_path, 'w', encoding='utf-8') as f:
    f.write(code)

print('[patched] config.py:')
print(subprocess.run(
    ['grep', '-nE',
     r'^\s*(repeats|eval_interval|full_eval_interval|save_interval):',
     cfg_path],
    capture_output=True, text=True).stdout)

# 后续 cell 用的全局变量
TRAIN_SCRIPT = os.path.join(DST_CODE, 'train_unet.py')
SPLITS_JSON  = os.path.join(DST_CODE, 'splits.json')
SAVE_DIR     = '/kaggle/working/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'TRAIN_SCRIPT = {TRAIN_SCRIPT}')
print(f'SPLITS_JSON  = {SPLITS_JSON}')
print(f'SAVE_DIR     = {SAVE_DIR}')

In [ ]:
# === Cell 4: 启动训练（subprocess + 日志重定向到文件，避免 notebook 输出灌爆）===
import subprocess, sys, time

if SMOKE_TEST:
    EPOCHS, BS, NW = 5, 4, 2
    extra = ['--smoke_test']
else:
    EPOCHS, BS, NW = 60, 8, 4
    extra = []

cmd = [
    sys.executable, TRAIN_SCRIPT,
    '--splits_json', SPLITS_JSON,
    '--data_dir',    DATA_DIR,
    '--save_dir',    SAVE_DIR,
    '--epochs',      str(EPOCHS),
    '--batch_size',  str(BS),
    '--num_workers', str(NW),
] + extra

LOG = '/kaggle/working/train_log_full.txt'
print('[cmd]', ' '.join(cmd))
print(f'[log] full stdout -> {LOG}')
print(f'[mode] {"SMOKE" if SMOKE_TEST else "FULL"}: epochs={EPOCHS} bs={BS} workers={NW}')
print('[note] 训练期间用 !tail -n 30 ' + LOG + ' 查看进度')
print('-' * 70)

t0 = time.time()
with open(LOG, 'w', encoding='utf-8') as f:
    ret = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
dt = (time.time() - t0) / 60
print(f'[done] return code = {ret.returncode}, elapsed = {dt:.1f} min')

# 训练完总是 tail 最后 80 行，便于排错
print('\n=== last 80 lines of train log ===')
print(subprocess.run(['tail', '-n', '80', LOG], capture_output=True, text=True).stdout)

if ret.returncode != 0:
    raise SystemExit(f'[FATAL] 训练失败，return code = {ret.returncode}')

In [ ]:
# === Cell 5: 产物校验 ===
import os, torch

EMA_PATH = os.path.join(SAVE_DIR, 'unet_best_ema.pth')
assert os.path.exists(EMA_PATH), f'[FATAL] 找不到 {EMA_PATH}'

size_mb = os.path.getsize(EMA_PATH) / 1e6
ckpt = torch.load(EMA_PATH, map_location='cpu', weights_only=False)

print(f'=== unet_best_ema.pth metadata ({size_mb:.1f} MB) ===')
for k in ('best_epoch', 'last_epoch', 'best_dice', 'weights_kind',
          'split_fingerprint', 'data_dir', 'timestamp', 'git_commit',
          'produced_by'):
    print(f'  {k:20s} = {ckpt.get(k)!r}')

# 红线校验
assert ckpt.get('split_fingerprint') == '84706e8b75c8f403', 'fingerprint 不匹配'
assert ckpt.get('weights_kind') in ('ema', 'raw'),         'weights_kind 字段缺失'
assert 'model_state_dict' in ckpt,                         'model_state_dict 缺失'
for forbidden in ('optimizer_state_dict', 'scheduler_state_dict',
                  'scaler_state_dict', 'ema_state_dict'):
    assert forbidden not in ckpt, f'禁含字段仍存在：{forbidden}'
assert size_mb < 200, f'体积异常 ({size_mb:.1f} MB)，应 < 150 MB'

print('\n[OK] 全部校验通过')

# 列出 checkpoints/ 全部产物
import subprocess
print('\n=== checkpoints 目录 ===')
print(subprocess.run(['ls', '-lah', SAVE_DIR], capture_output=True, text=True).stdout)

In [ ]:
# === Cell 6: 打包成 unet_output.zip 方便下载 ===
import shutil, os

OUT_ZIP = '/kaggle/working/unet_output'
STAGE   = '/kaggle/working/_unet_stage'

if os.path.exists(STAGE):
    shutil.rmtree(STAGE)
os.makedirs(STAGE, exist_ok=True)

# 只打包推理需要的 + 论文需要的产物
for fn in ('unet_best_ema.pth', 'training_curves.png', 'training_log.csv'):
    src = os.path.join(SAVE_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(STAGE, fn))
        print(f'[stage] {fn}  ({os.path.getsize(src)/1e6:.1f} MB)')
    else:
        print(f'[skip ] {fn} 不存在')

# 训练日志一并打包，方便事后排查
if os.path.exists('/kaggle/working/train_log_full.txt'):
    shutil.copy('/kaggle/working/train_log_full.txt',
                os.path.join(STAGE, 'train_log_full.txt'))

zip_path = shutil.make_archive(OUT_ZIP, 'zip', STAGE)
print(f'\n[zip] {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('\n下载步骤：右侧 Output 面板 → 找到 unet_output.zip → 点 Download')
print('解压后把 unet_best_ema.pth 放到本地 ctai_model_code/CTAI_model/checkpoints/')